# T4 inference comparison
CPU-stage the established 180k model and the 42M 6k finalist. Verify both bundles before allocating a T4. Compare FP32 batches 64, 128 and 256 twice, preserving exact outputs and timing separately from Drive export. No training.

In [ ]:
from pathlib import Path
assert not Path('/dev/nvidia0').exists(), 'Select CPU first'
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
import os, subprocess, sys
os.environ['USE_TF']='0'
os.environ['USE_FLAX']='0'
repo=Path('/content/hse-nlp-t4-comparison')
subprocess.run(['git','clone','--depth=1','--branch=2026','https://github.com/OsVox/hse-nlp.git',str(repo)],check=True)
os.chdir(repo)
subprocess.run([sys.executable,'-m','pip','-q','install','transformers==4.57.6','pyarrow>=18','requests>=2.31'],check=True)
from homeworks.hw2.t4_comparison import prepare
prepare()


Export and verify prepared/t4-comparison-v1/ready.json from Drive. Only then select one NVIDIA T4. Do not Run All.

In [ ]:
from google.colab import drive, runtime
try:
    drive.mount('/content/drive', timeout_ms=90000)
    import os, sys, json, hashlib, tarfile, subprocess, asyncio, signal, time
    from pathlib import Path
    prepared=Path('/content/drive/MyDrive/HSE_NLP_HW2/prepared/t4-comparison-v1')
    ready=json.loads((prepared/'ready.json').read_text())
    assert ready['cpu_verified']
    payload=(prepared/'source.tar').read_bytes()
    assert hashlib.sha256(payload).hexdigest()==ready['source_sha256']
    work=Path('/content/hw2-t4-comparison')
    work.mkdir(exist_ok=False)
    os.chdir(work)
    Path('source.tar').write_bytes(payload)
    with tarfile.open('source.tar') as archive: archive.extractall('.',filter='data')
    subprocess.run([sys.executable,'-m','pip','-q','install','transformers==4.57.6','pyarrow>=18','requests>=2.31'],check=True,timeout=180)
    os.environ['USE_TF']='0'
    os.environ['USE_FLAX']='0'
    pilot_log=(prepared/'execution.log').open('w')
    pilot_process=subprocess.Popen([sys.executable,'-u','-m','homeworks.hw2.t4_comparison','--gpu'],stdout=pilot_log,stderr=subprocess.STDOUT,start_new_session=True)
    async def finish_pilots():
        status={'pid':pilot_process.pid,'started_utc':time.time()}
        try:
            await asyncio.wait_for(asyncio.to_thread(pilot_process.wait),timeout=1800)
            status['returncode']=pilot_process.returncode
        except BaseException as error:
            status['error']=type(error).__name__+': '+str(error)
            if pilot_process.poll() is None:
                os.killpg(pilot_process.pid,signal.SIGTERM)
                try: await asyncio.wait_for(asyncio.to_thread(pilot_process.wait),timeout=10)
                except asyncio.TimeoutError: os.killpg(pilot_process.pid,signal.SIGKILL)
        finally:
            pilot_log.close()
            status['finished_utc']=time.time()
            try: (prepared/'execution.complete.json').write_text(json.dumps(status,indent=2))
            finally: runtime.unassign()
    pilot_task=asyncio.create_task(finish_pilots())
    print('PILOTS_STARTED',pilot_process.pid,'Automatic notebook-kernel release is armed.')
except BaseException:
    runtime.unassign()
    raise


Verify GPU release. Export results from Drive. Prefer a consistently faster batch, and compare exact output hashes. If the selected T4 output differs from a previously scored export, submit that exact output to the bot and save its receipt. Timing a pilot does not establish timing for later weights.